# The Fleet App — `amygda_ops_risk_score`

## What this guide covers

Continues from `02_risk_score_guide.ipynb`. Once you have a model you are happy with, the
**fleet app** keeps scoring new logs with it. It is a small app on your own machine:

- a page in your browser that **shows what you choose**: tabs, panels, colours, your logo;
- buttons to **add files**, **run now**, and **set, pause or resume a schedule**;
- **alerts** when something you care about happens.

Everything is done with the `amygda-risk` command, which is installed with the SDK. This
guide lists the commands in order. There is nothing to code. To try them on the sample
data, run `examples/03a_fleet_app_fixed_log.ipynb` or `03b_fleet_app_free_text.ipynb`.

| Step | What you do |
|---|---|
| 1 | Before you start |
| 2 | See what the app can show |
| 3 | Write your layout |
| 4 | Preview it |
| 5 | Set up the app |
| 6 | Open it and add your data |
| 7 | Make it start by itself |
| 8 | Alerts |
| 9 | Change things later |
| 10 | Remove it |
| 11 | Or let the agent skill do it |

---
## 1. Before you start

You need two things.

**A complete model zip.** `generate_risk_scores()` (Step 12) saves it as
`complete_model_<id>.zip`.

**Your API key in a file named `.env`**, in the folder you work in:

```
AMYGDA_API_KEY=your_key_here
```

The commands and the app read the key from this file. It is never copied anywhere else.

---
## 2. See what the app can show

```bash
amygda-risk panels
```

This lists every **panel** (a chart or table, such as the ranking of assets or the alerts
list) and every **tile** (a figure at the top of the page, such as the number of open
alerts), with a line on what each shows.

---
## 3. Write your layout

The layout is one small file. Save it as, say, `my_app.json`:

```json
{
  "title": "Conveyor fleet risk",
  "theme": {"mode": "auto", "accent": "#0b6e4f", "logo": "my_logo.png"},
  "header": {"tiles": ["open_alerts", "assets_high_now", "highest", "last_run"]},
  "tabs": [
    {"title": "Fleet",  "panels": ["since_last_run", "fleet_systems", "ranking"]},
    {"title": "Alerts", "panels": ["alerts"]},
    {"title": "Assets", "panels": ["asset_trend", "asset_systems", "asset_drivers"]},
    {"title": "Runs",   "panels": ["run_health", "run_table"]}
  ],
  "alerts": {"rules": [
    {"type": "enters_band", "band": "high"},
    {"type": "rises_by", "points": 10},
    {"type": "run_failed", "runs": 1}
  ]},
  "schedule": {"every": "day", "at": "06:00"}
}
```

| Setting | What it does |
|---|---|
| `title` | The name at the top of the page, exactly as you write it |
| `theme` | `mode`: `auto`, `light` or `dark`. `accent`: your brand colour. `logo`: an image file next to this file. All optional |
| `header.tiles` | The figures at the top, up to 6 |
| `tabs` | Each tab has a `title` and its `panels`, top to bottom. The first tab opens first |
| `alerts.rules` | What raises an alert (step 8) |
| `schedule` | How often the app checks for new files: `{"every": "hours", "hours": 6}`, `{"every": "day", "at": "06:00"}` or `{"every": "week", "weekday": "mon", "at": "06:00"}` |

Everything is optional. Leave `tabs` out and you get the standard layout: Fleet, Asset
detail, Alerts, Model and Runs. If the file has a mistake, the next command says what is
wrong in plain words.

---
## 4. Preview it

```bash
amygda-risk design --spec my_app.json --model complete_model_<id>.zip --open
```

This builds the page from the scores saved inside the model zip and opens it in your
browser. Nothing is deployed and nothing is sent anywhere. Panels that need a run (what
changed, alerts, run history) show as "pending" cards, so you can see where they will be.

Change `my_app.json` and run the command again until you like it.

---
## 5. Set up the app

Two commands. The first deploys the model and creates a folder for the app's data. The
second applies your layout.

```bash
amygda-risk init fleet_monitor --model complete_model_<id>.zip --name conveyor-fleet-v1 \
    --history training_logs.csv --fleet-name "Conveyor fleet"

amygda-risk design fleet_monitor --spec my_app.json
```

| Option | What it does |
|---|---|
| `fleet_monitor` | The folder to create. Your scores, run records and alerts live here |
| `--name` | A name for the deployed model |
| `--history` | Earlier logs, usually the training file. The model needs some history before the first new file, or the first scores come out low |
| `--watch 35 --high 45` | Your watch and high levels. Left out, they are set from the model's training scores (95th and 99th percentile) |
| `--storage sqlite` | Keep the scores in one database file (`monitor.db`) instead of ordinary files |
| `--api-key-env NAME` | If your `.env` calls the key something other than `AMYGDA_API_KEY` |

Setting up scores nothing. For a fixed-log model, `init` warns you if many of the training
logs have codes the model doesn't use.

Deploying keeps a small scoring bundle on Amygda's servers, never your log rows. The SDK
README has the details under "Data retention".

---
## 6. Open it and add your data

```bash
amygda-risk app fleet_monitor
```

The app opens at `http://127.0.0.1:8050/` and says it is waiting for data. It runs until
you press Ctrl+C. It listens on this machine only and has no login.

On the page:

- **Add files**: choose a CSV, XLSX or parquet file of new logs, in the same columns as
  your training file. You can also drop files into `fleet_monitor/inbox/`.
- **Run now**: scores the waiting files. The page fills in when the run finishes.
- **Schedule**: change how often it checks for new files, or pause and resume it.

Each run is listed on the Runs tab with the dates its data covered. A file you add twice
is not scored twice. If a run fails, the page says why, and the file waits for the next
run.

---
## 7. Make it start by itself

The schedule only works while the app is running. This starts the app whenever you log in
to this computer, and restarts it if it stops:

```bash
amygda-risk service fleet_monitor --name conveyor-fleet --install
```

`--name` is your own name for it. To see what would be installed without changing
anything, leave out `--install`. To stop it starting by itself:

```bash
amygda-risk service fleet_monitor --remove
```

The computer must be on and you must be logged in for a scheduled check. A check missed
while it was off is made when the app next starts.

---
## 8. Alerts

Alert rules go in the layout file (step 3) and are checked after every run.

| Rule | Raises an alert when |
|---|---|
| `{"type": "enters_band", "band": "high"}` | An asset's score reaches high (`"watch"` covers watch and high) |
| `{"type": "rises_by", "points": 10}` | An asset's score rises by 10 points or more in one run |
| `{"type": "no_data", "days": 7}` | An asset has had no logs for 7 days while others have |
| `{"type": "run_failed", "runs": 1}` | A run fails |
| `{"type": "unmatched_above", "percent": 30}` | More than 30% of a run's new logs match nothing the model knows |

Alerts show on the Alerts panel, where you acknowledge them with a click. They are not
emailed or sent anywhere. From the command line:

```bash
amygda-risk alerts fleet_monitor              # open alerts
amygda-risk alerts fleet_monitor --ack all    # acknowledge them
```

---
## 9. Change things later

| You want | Command |
|---|---|
| A different layout, look or alert rules | Edit `my_app.json`, then `amygda-risk design fleet_monitor --spec my_app.json`. The open page updates by itself |
| The standard layout back | `amygda-risk design fleet_monitor --reset` |
| A different schedule | The **Schedule** button on the page |
| A summary of where things stand | `amygda-risk status fleet_monitor` |
| To score a file from the command line | `amygda-risk run fleet_monitor --input new_logs.csv` |
| To try a file without keeping the result | `amygda-risk score fleet_monitor --input other.csv` |
| To switch between files and a database | `amygda-risk storage fleet_monitor --to sqlite` (or `files`), with the app stopped |

---
## 10. Remove it

```bash
amygda-risk remove fleet_monitor          # lists what it would do; changes nothing
amygda-risk remove fleet_monitor --yes    # does it
```

This stops the app, deletes the deployed model from the API and deletes the folder with
its scores and history. It can't be undone. Add `--keep-deployment` to leave the deployed
model in place.

---
## 11. Or let the agent skill do it

With the SDK's agent skill installed (`amygda-risk-skill install`), ask your coding agent
for a fleet monitoring app. It asks what the app is called, which panels and tabs you
want, who will use it, how it should look, what should raise an alert and how often to
check for new files. It then writes the layout, shows you the preview, sets everything up
and opens the app, waiting for your data.